In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/model_ready_dataset.csv")

In [3]:
df.shape


(3756, 25)

In [4]:
df.columns.tolist()

['location_name',
 'sensor_id',
 'datetime',
 'parameter',
 'pm25',
 'coverage.percentComplete',
 'coverage.percentCoverage',
 'hour',
 'hour_utc',
 'temperature_2m',
 'relative_humidity_2m',
 'wind_speed_10m',
 'wind_direction_10m',
 'precipitation',
 'hour_of_day',
 'day_of_week',
 'month',
 'is_weekend',
 'pm25_lag_1',
 'pm25_lag_3',
 'pm25_lag_6',
 'pm25_rolling_3',
 'pm25_rolling_6',
 'pm25_rolling_12',
 'pm25_24h_ahead']

In [5]:
df["location_name"].value_counts()

location_name
R K Puram, Delhi - DPCC          1281
Punjabi Bagh, Delhi - DPCC       1248
Anand Vihar, New Delhi - DPCC    1227
Name: count, dtype: int64

In [6]:
df["hour_utc"] = pd.to_datetime(df["hour_utc"], utc=True)
df = df.sort_values(["location_name", "hour_utc"]).reset_index(drop=True)

In [7]:
df.groupby("location_name")["hour_utc"].agg(["min", "max", "count"])


,min,max,count
location_name,,,
"Anand Vihar, New Delhi - DPCC",2026-04-05 07:00:00+00:00,2026-06-02 15:00:00+00:00,1227
"Punjabi Bagh, Delhi - DPCC",2026-04-05 07:00:00+00:00,2026-06-02 15:00:00+00:00,1248
"R K Puram, Delhi - DPCC",2026-04-05 07:00:00+00:00,2026-06-02 15:00:00+00:00,1281


In [8]:
# Position of each row within its station's timeline (0, 1, 2, ...)
df["row_in_station"] = df.groupby("location_name").cumcount()

# Total rows for that station
df["station_size"] = df.groupby("location_name")["location_name"].transform("count")

# First 80% per station = train, last 20% = test
df["split"] = (df["row_in_station"] < 0.8 * df["station_size"]).map({True: "train", False: "test"})

train = df[df["split"] == "train"].copy()
test  = df[df["split"] == "test"].copy()

In [9]:
df.groupby(["location_name", "split"])["hour_utc"].agg(["min", "max", "count"])

min  \
location_name                 split                             
Anand Vihar, New Delhi - DPCC test  2026-05-21 13:00:00+00:00   
                              train 2026-04-05 07:00:00+00:00   
Punjabi Bagh, Delhi - DPCC    test  2026-05-21 21:00:00+00:00   
                              train 2026-04-05 07:00:00+00:00   
R K Puram, Delhi - DPCC       test  2026-05-22 15:00:00+00:00   
                              train 2026-04-05 07:00:00+00:00   

                                                          max  count  
location_name                 split                                   
Anand Vihar, New Delhi - DPCC test  2026-06-02 15:00:00+00:00    245  
                              train 2026-05-21 12:00:00+00:00    982  
Punjabi Bagh, Delhi - DPCC    test  2026-06-02 15:00:00+00:00    249  
                              train 2026-05-21 20:00:00+00:00    999  
R K Puram, Delhi - DPCC       test  2026-06-02 15:00:00+00:00    256  
                              train 2026-05-22 14:00:00+00:00   1025

In [11]:
from sklearn.metrics import mean_absolute_error

# Persistence prediction: tomorrow = today
test["baseline_pred"] = test["pm25"]

# Make sure there are no NaNs in target (sanity)
assert test["pm25_24h_ahead"].isna().sum() == 0

In [12]:
from sklearn.metrics import mean_absolute_error

# Overall baseline MAE
overall_mae = mean_absolute_error(test["pm25_24h_ahead"], test["baseline_pred"])
print(f"Overall baseline MAE: {overall_mae:.2f} µg/m³")

# Per-station baseline MAE
per_station_mae = (
    test.groupby("location_name")
        .apply(lambda g: mean_absolute_error(g["pm25_24h_ahead"], g["baseline_pred"]))
        .rename("baseline_mae")
)
print(per_station_mae)

Overall baseline MAE: 26.59 µg/m³
location_name
Anand Vihar, New Delhi - DPCC    32.933878
Punjabi Bagh, Delhi - DPCC       21.630924
R K Puram, Delhi - DPCC          25.342187
Name: baseline_mae, dtype: float64


In [13]:
# Save the test set with baseline predictions 
test.to_csv("../data/processed/test_with_baseline.csv", index=False)